# Subsystem Discovery Notebook

This notebook implements the subsystem discovery process for the Legacy MUD codebase. It uses multi-dimensional clustering approaches to identify coherent subsystems in the code.

## Setup and Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import json
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from loguru import logger as log
from pathlib import Path
from collections import defaultdict, Counter
from typing import Set, Dict, List, Tuple, Any

PROJECT_ROOT = Path('..').resolve()  # Adjust as needed to find the project root

# Add the modules directory to the Python path if needed
modules_dir = PROJECT_ROOT / 'modules'
if str(modules_dir) not in sys.path:
    sys.path.append(str(modules_dir))

import subsystem_utils as su
import structural_clustering as sc
import semantic_clustering as sec
import cluster_integration as ci

## Load Resources

In [ ]:
# Define paths
GRAPH_PATH = su.DEFAULT_GRAPH_PATH
DB_PATH = su.DEFAULT_DB_PATH

# Create directories if they don't exist
os.makedirs(su.INTERNAL_DIR, exist_ok=True)

# Load graph - graph is now a MultiDiGraph
log.info(f"Loading graph from {GRAPH_PATH}...")
graph = su.load_graph(GRAPH_PATH)
log.info(f"Loaded graph with {len(graph.nodes())} nodes and {len(graph.edges())} edges.")

# Load entity database
log.info(f"Loading entity database from {DB_PATH}...")
entity_db = su.load_entity_db(DB_PATH)
log.info(f"Loaded entity database with {len(entity_db.entities)} entries.")

# Load document database
log.info("Loading document database...")
from doc_db import docs_db
log.info(f"Loaded document database with {len(docs_db)} compounds.")

# Print edge types distribution to see the relationship types present in the graph
edge_types = {}
for u, v, data in graph.edges(data=True):
    edge_type = data.get('type', 'unknown')
    edge_types[edge_type] = edge_types.get(edge_type, 0) + 1

print("\nEdge type distribution:")
for edge_type, count in sorted(edge_types.items(), key=lambda x: x[1], reverse=True):
    print(f"{edge_type}: {count} edges ({count/len(graph.edges())*100:.2f}%)")

2026-03-11 21:54:58.210 | INFO     | __main__:<module>:9 - Loading graph from /Users/QTE2333/repos/legacy-entity-docs/context/internal/code_graph.gml...
2026-03-11 21:54:59.250 | INFO     | doxygen_graph:load_graph:608 - Graph loaded from /Users/QTE2333/repos/legacy-entity-docs/context/internal/code_graph.gml, nodes: 14507, edges: 44544
2026-03-11 21:54:59.329 | INFO     | __main__:<module>:11 - Loaded graph with 4549 nodes and 29174 edges.
2026-03-11 21:54:59.330 | INFO     | __main__:<module>:14 - Loading entity database from /Users/QTE2333/repos/legacy-entity-docs/context/internal/code_graph.json...
2026-03-11 21:54:59.580 | INFO     | __main__:<module>:16 - Loaded entity database with 5305 entries.
2026-03-11 21:54:59.580 | INFO     | __main__:<module>:19 - Loading document database...
2026-03-11 21:54:59.581 | INFO     | __main__:<module>:21 - Loaded document database with 475 entries.



Edge type distribution:
uses: 16681 edges (57.18%)
calls: 9875 edges (33.85%)
contained_by: 2565 edges (8.79%)
inherits: 53 edges (0.18%)


In [18]:
print(docs_db.docs.keys())

dict_keys(['mobiles_8cc', 'event_8hh', 'structgem_1_1type__st', 'group___loot_armor_vnums', 'ignore_8cc', 'structcon__app__type', 'group___gem_object_primitives', '_exit_8cc', 'structconn_1_1_roll_stats_state', 'group___video_mode_flags', 'group___damage_type_vulnerability_flags', 'structaffect_1_1fn__data__container__type', 'group___size_constants', 'structhunt__conditions', 'config_8cc', 'dir_1f5ef0dbc3555222c9e1262e058d8011', 'structconn_1_1_break_connect_state', 'structchan__type', 'structsector__type', 'group___tail_flags', 'group___ban_flags', 'structaffect__table__type', 'group___flag_candidates', '_mob_prog_8hh', 'gem_8cc', '_weather_8hh', 'clan-edit_8cc', 'structprintbuffer', '_dispatcher_8cc', '_get_alignment_state_8cc', '_location_8hh', 'skill_8hh', '_coordinate_8hh', 'affect__room_8cc', 'remort_8cc', 'class_object_value', '_vnum_8hh', 'sqlite_8cc', 'lookup_8cc', 'lootv2_8hh', 'structeq__quality__t', 'namespaceevent', 'typename_8cc', 'classworldmap_1_1_quadtree', '_character

## Phase 1: Multi-Dimensional Clustering

### 1.1 Structural Clustering

Apply community detection algorithms to the dependency graph.

In [4]:
# Run structural clustering using Leiden algorithm
log.info("Running structural clustering...")
etypes = ('calls', 'uses')

def calc_utility_score(score: Dict[str, float]) -> bool:
    return (
        ((
            sum(score[f'fan_in_{etype}'] for etype in etypes) +
            sum(score[f'fan_in_clusters_{etype}'] for etype in etypes) +
            0
        )/2) 
        * 
        (1-(
            sum(score[f'fan_out_{etype}'] for etype in etypes) + # calls, uses, inherits
            sum(score[f'fan_out_clusters_{etype}'] for etype in etypes) +
            sum(score[f'overlap_{etype}'] for etype in etypes) + # overlap - higher value means more used outside its cluster
            0
        )/2)
    )

structural_clustering, structural_metadata = sc.run_structural_clustering(
    graph,
    utility_threshold_fn=lambda score: calc_utility_score(score) > 0.226,
    filter_utilities=False, #True,
    resolution=1.0
)
log.info(f"Generated {structural_metadata['refined_cluster_count']} structural clusters.")

# Convert to cluster objects
structural_clusters = sc.clustering_to_cluster_objects(structural_clustering)

2026-03-11 21:54:59.619 | INFO     | __main__:<module>:2 - Running structural clustering...
2026-03-11 21:54:59.624 | INFO     | structural_clustering:run_structural_clustering:312 - Starting structural clustering on graph with 4549 nodes and 29174 edges
2026-03-11 21:54:59.625 | INFO     | structural_clustering:run_structural_clustering:313 - Configuration: algorithm=leiden, filter_utilities=False, resolution=1.0
2026-03-11 21:54:59.625 | INFO     | structural_clustering:run_structural_clustering:322 - Stage 1: Initial clustering (including utility nodes)
2026-03-11 21:54:59.688 | INFO     | structural_clustering:apply_leiden_clustering:134 - Applying Leiden clustering with resolution=1.0 on graph with 4549 nodes and 29159 edges
2026-03-11 21:54:59.720 | INFO     | structural_clustering:apply_leiden_clustering:152 - Filtered graph created with 4153 nodes and 26594 edges
2026-03-11 21:54:59.721 | INFO     | structural_clustering:apply_leiden_clustering:155 - Converting NetworkX graph t

In [5]:
# Save structural clusters
STRUCTURAL_CLUSTERS_PATH = su.INTERNAL_DIR / "structural_clusters.json"
su.save_clusters(structural_clusters, STRUCTURAL_CLUSTERS_PATH, metadata=structural_metadata)
print(f"Saved structural clusters to {STRUCTURAL_CLUSTERS_PATH}")

Saved structural clusters to /Users/QTE2333/repos/legacy-entity-docs/context/internal/structural_clusters.json


In [6]:
# Basic statistics on structural clusters
print(f"Number of structural clusters: {len(structural_clusters)}")
cluster_sizes = [len(c.nodes) for c in structural_clusters.values()]
print(f"Average cluster size: {np.mean(cluster_sizes):.2f} entities")
print(f"Minimum cluster size: {min(cluster_sizes)} entities")
print(f"Maximum cluster size: {max(cluster_sizes)} entities")
print(f"Modularity: {structural_metadata.get('modularity', 'N/A')}")

Number of structural clusters: 25
Average cluster size: 166.12 entities
Minimum cluster size: 2 entities
Maximum cluster size: 798 entities
Modularity: N/A


### 1.2 Semantic Clustering

Cluster entities based on their documentation text.

In [7]:
# Run semantic clustering
print("Running semantic clustering...")
semantic_clustering, semantic_metadata = sec.run_semantic_clustering(
    graph,
    entity_db,
    docs_db,
    topic_method="nmf",  # Use NMF as it often performs better for short texts
    n_topics=15,         # Start with a moderate number of topics
    cluster_method="kmeans",
    n_clusters=26
)
print(f"Generated {semantic_metadata.get('n_clusters', 0)} semantic clusters.")

# Convert to cluster objects
semantic_clusters = sec.clustering_to_cluster_objects(semantic_clustering)

2026-03-11 21:55:00.128 | INFO     | semantic_clustering:run_semantic_clustering:350 - Starting semantic clustering with nmf topic modeling and kmeans clustering


2026-03-11 21:55:00.128 | INFO     | semantic_clustering:extract_entity_texts:68 - Extracting textual content from document database
2026-03-11 21:55:00.250 | INFO     | semantic_clustering:extract_entity_texts:103 - Extracted text from 4549 documents in 0.12s
2026-03-11 21:55:00.251 | INFO     | semantic_clustering:create_document_term_matrix:121 - Creating document-term matrix with TF-IDF vectorization (min_df=0.01, max_df=0.95)
2026-03-11 21:55:00.251 | INFO     | semantic_clustering:create_document_term_matrix:128 - Creating TF-IDF vectorizer for 4549 documents


Running semantic clustering...


2026-03-11 21:55:00.540 | INFO     | semantic_clustering:create_document_term_matrix:147 - Created document-term matrix with shape (4549, 1617) and 1617 features in 0.29s
2026-03-11 21:55:01.158 | INFO     | semantic_clustering:run_semantic_clustering:414 - Semantic clustering completed in 1.03s. Found 26 clusters
2026-03-11 21:55:01.159 | INFO     | semantic_clustering:clustering_to_cluster_objects:430 - Converting 4549 nodes from semantic clustering to Cluster objects
2026-03-11 21:55:01.160 | INFO     | semantic_clustering:clustering_to_cluster_objects:448 - Created 26 cluster objects with size distribution: min=37, max=462, avg=175.0


Generated 26 semantic clusters.


In [8]:
# Save semantic clusters
SEMANTIC_CLUSTERS_PATH = su.INTERNAL_DIR / "semantic_clusters.json"
su.save_clusters(semantic_clusters, SEMANTIC_CLUSTERS_PATH, metadata=semantic_metadata)
print(f"Saved semantic clusters to {SEMANTIC_CLUSTERS_PATH}")

Saved semantic clusters to /Users/QTE2333/repos/legacy-entity-docs/context/internal/semantic_clusters.json


In [9]:
# Basic statistics on semantic clusters
print(f"Number of semantic clusters: {len(semantic_clusters)}")
cluster_sizes = [len(c.nodes) for c in semantic_clusters.values()]  # Using nodes attribute
print(f"Average cluster size: {np.mean(cluster_sizes):.2f} entities")
print(f"Minimum cluster size: {min(cluster_sizes)} entities")
print(f"Maximum cluster size: {max(cluster_sizes)} entities")
print(f"Silhouette score: {semantic_metadata.get('silhouette_score', 'N/A')}")

# List topics if available
if 'topic_labels' in semantic_metadata:
    print("\nTop topics:")
    for topic in semantic_metadata['topic_labels'][:5]:
        print(f"- {topic}")

Number of semantic clusters: 26
Average cluster size: 174.96 entities
Minimum cluster size: 37 entities
Maximum cluster size: 462 entities
Silhouette score: 0.28154304539026853

Top topics:
- Topic 0: character function player message command
- Topic 1: flag bit flags bit flag entity
- Topic 2: json item cjson memory data
- Topic 3: constant game consistent identifier clear
- Topic 4: object constructor copy new assignment


### 1.4 Integrated Clustering

Combine the results from all clustering approaches.

In [10]:
# Define weights for different clustering approaches
weights = {
    "structural": 0.5,  # Structure is most important
    "semantic": 0.3,    # Semantics are valuable but less reliable
    "usage": 0.2        # Usage patterns are useful but may be noisy
}

# Run integrated clustering
print("Running integrated clustering...")
integrated_clustering, integrated_metadata = ci.run_cluster_integration(
    structural_clustering=structural_clustering,
    semantic_clustering=semantic_clustering,
    weights=weights,
    n_clusters=26
)
print(f"Generated {integrated_metadata.get('cluster_count', 0)} integrated clusters.")

# Convert to cluster objects
integrated_clusters = ci.clustering_to_cluster_objects(integrated_clustering, source="integrated")

Running integrated clustering...
Generated 26 integrated clusters.


In [11]:
# Save integrated clusters
INTEGRATED_CLUSTERS_PATH = su.INTERNAL_DIR / "integrated_clusters.json"
su.save_clusters(integrated_clusters, INTEGRATED_CLUSTERS_PATH, metadata=integrated_metadata)
print(f"Saved integrated clusters to {INTEGRATED_CLUSTERS_PATH}")

Saved integrated clusters to /Users/QTE2333/repos/legacy-entity-docs/context/internal/integrated_clusters.json


In [12]:
# Basic statistics on integrated clusters
print(f"Number of integrated clusters: {len(integrated_clusters)}")
cluster_sizes = [len(c.nodes) for c in integrated_clusters.values()]  # Using nodes attribute
print(f"Average cluster size: {np.mean(cluster_sizes):.2f} entities")
print(f"Minimum cluster size: {min(cluster_sizes)} entities")
print(f"Maximum cluster size: {max(cluster_sizes)} entities")

# Agreement scores between clustering approaches
if 'agreement_scores' in integrated_metadata:
    print("\nAgreement between clustering approaches:")
    for pair, score in integrated_metadata['agreement_scores'].items():
        print(f"{pair}: {score:.4f}")

Number of integrated clusters: 26
Average cluster size: 174.96 entities
Minimum cluster size: 1 entities
Maximum cluster size: 588 entities

Agreement between clustering approaches:
structural_semantic: 0.3086


## Generate Clustering Reports

In [13]:
# Function to generate a cluster summary for a report
from typing import Dict


def generate_cluster_summary(graph, clusters: Dict[su.ClusterID, su.Cluster], entity_db, max_nodes=10):
    report = []
    
    for cluster_id, cluster in sorted(clusters.items()):
        report.append(f"### Cluster {cluster_id}\n")
        report.append(f"**Size**: {len(cluster.nodes)} nodes\n")
        
        # List some representative nodes
        nodes = sorted(cluster.nodes)
        sample = nodes[:max_nodes]
        
        report.append("**Representative nodes**:\n")
        for node_id in sample:
            node_name = graph.nodes[node_id].get('name', node_id)
            node_type = graph.nodes[node_id].get('type', 'unknown')
            report.append(f"- `{node_name}` ({node_type})")

        if len(nodes) > max_nodes:
            report.append(f"- *...and {len(nodes) - max_nodes} more*")
            
        report.append("\n")
        
    return "\n".join(report)

# Generate structural clustering report
print("Generating structural clustering report...")
structural_report = [
    "# Structural Clustering Report\n",
    "## Overview\n",
    f"Total clusters: {len(structural_clusters)}\n",
    f"Total entities: {sum(len(c.nodes) for c in structural_clusters.values())}\n",
    f"Algorithm: {structural_metadata.get('algorithm', 'unknown')}\n",
    f"Modularity: {structural_metadata.get('modularity', 'N/A')}\n",
    "\n## Cluster Analysis\n",
    generate_cluster_summary(graph, structural_clusters, entity_db)
]

# Save report
STRUCTURAL_REPORT_PATH = su.DOCS_DIR / "structural_clustering_report.md"
su.create_markdown_report(
    "Structural Clustering Report", 
    "\n".join(structural_report),
    STRUCTURAL_REPORT_PATH
)
print(f"Saved structural clustering report to {STRUCTURAL_REPORT_PATH}")

Generating structural clustering report...
Saved structural clustering report to /Users/QTE2333/repos/legacy-entity-docs/docs/structural_clustering_report.md


In [14]:
# Generate semantic clustering report
print("Generating semantic clustering report...")
semantic_report = [
    "# Semantic Clustering Report\n",
    "## Overview\n",
    f"Total clusters: {len(semantic_clusters)}\n",
    f"Total entities: {sum(len(c.nodes) for c in semantic_clusters.values())}\n",
    f"Topic method: {semantic_metadata.get('topic_method', 'unknown')}\n",
    f"Number of topics: {semantic_metadata.get('n_topics', 'N/A')}\n",
    f"Silhouette score: {semantic_metadata.get('silhouette_score', 'N/A')}\n",
]

# Add topic information if available
if 'topic_labels' in semantic_metadata:
    semantic_report.append("\n## Topics\n")
    for i, topic in enumerate(semantic_metadata['topic_labels']):
        semantic_report.append(f"{i+1}. {topic}\n")
        
semantic_report.append("\n## Cluster Analysis\n")
semantic_report.append(generate_cluster_summary(graph, semantic_clusters, entity_db))

# Save report
SEMANTIC_REPORT_PATH = su.DOCS_DIR / "semantic_clustering_report.md"
su.create_markdown_report(
    "Semantic Clustering Report", 
    "\n".join(semantic_report),
    SEMANTIC_REPORT_PATH
)
print(f"Saved semantic clustering report to {SEMANTIC_REPORT_PATH}")

Generating semantic clustering report...
Saved semantic clustering report to /Users/QTE2333/repos/legacy-entity-docs/docs/semantic_clustering_report.md


In [15]:
# Generate integrated clustering report
print("Generating integrated clustering report...")
integrated_report = [
    "# Integrated Clustering Report\n",
    "## Overview\n",
    f"Total clusters: {len(integrated_clusters)}\n",
    f"Total entities: {sum(len(c.nodes) for c in integrated_clusters.values())}\n",
    "\n## Integration Weights\n",
]

# Add weights
if 'weights' in integrated_metadata:
    for source, weight in integrated_metadata['weights'].items():
        integrated_report.append(f"- {source}: {weight:.2f}\n")

# Add agreement scores
if 'agreement_scores' in integrated_metadata:
    integrated_report.append("\n## Agreement Between Clustering Approaches\n")
    for pair, score in integrated_metadata['agreement_scores'].items():
        integrated_report.append(f"- {pair}: {score:.4f}\n")
        
integrated_report.append("\n## Cluster Analysis\n")
integrated_report.append(generate_cluster_summary(graph, integrated_clusters, entity_db))

# Save report
INTEGRATED_REPORT_PATH = su.DOCS_DIR / "integrated_clustering_report.md"
su.create_markdown_report(
    "Integrated Clustering Report", 
    "\n".join(integrated_report),
    INTEGRATED_REPORT_PATH
)
print(f"Saved integrated clustering report to {INTEGRATED_REPORT_PATH}")

Generating integrated clustering report...
Saved integrated clustering report to /Users/QTE2333/repos/legacy-entity-docs/docs/integrated_clustering_report.md


## Next Steps

The next phase is to name and classify the identified clusters, and organize them into a hierarchical model. This will be done in the subsystem identification phase.